# Assignment 1: Introduction to Language Modeling

**Deadline:** April 20

---

**Task markers:**
- 🎓 Suitable for oral exam discussion
- ⚙ Pure implementation task

---

## Part 0: Environment Setup

### Task 0.1 — Setting up the environment ⚙

Install the required libraries and download the provided data archive.

In [2]:
# !pip install nltk
# !pip install torch
# !pip install transformers
# !pip install datasets
# !pip install matplotlib
# !pip install scikit-learn

In [3]:
import torch
import torch.nn as nn
import numpy as np
from collections import Counter
from datasets import load_dataset
from torch.utils.data import DataLoader, Subset
from transformers import PretrainedConfig, PreTrainedModel

/home/hoda/anaconda3/envs/tch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


---

## Part 1: Tokenization

### Task 1.1 — Using NLTK or SpaCy for word splitting ⚙

Use `word_tokenize` from NLTK (or an equivalent SpaCy function) to split text into tokens.

Example: `word_tokenize("Let's test!!")` → `["Let", "'s", "test", "!", "!"]`

In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import word_tokenize

# Sanity check
print(word_tokenize("Let's test!!"))

[nltk_data] Downloading package punkt to /home/hoda/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/hoda/nltk_data...


['Let', "'s", 'test', '!', '!']


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


### Task 1.2 — Building the vocabulary 🎓

Process training and validation paragraphs with the tokenizer (lowercase).  
Build a vocabulary that maps tokens to integers, including **4 special symbols**:
- Unknown token
- Beginning-of-sequence token
- End-of-sequence token
- Padding token

Limit vocabulary to `max_voc_size` using the most frequent tokens.  
Also create an inverse mapping (integer → string).

**Sanity checks:**
- Verify vocabulary size
- Special symbols exist without conflicts
- Common and rare words are handled appropriately
- Bidirectional mapping (token → id → token) works correctly

In [ ]:
# Task 1.2 — Build vocabulary

max_voc_size = 10000

# Special tokens
UNK = "<unk>"
BOS = "<bos>"
EOS = "<eos>"
PAD = "<pad>"
#counter
counter = Counter()
for split in ("train", "val"):
    for item in dataset[split]:
        counter.update(word_tokenize(item['text'].lower()))
        
:# Build voc (dict: str -> int) and inv_voc (dict: int -> str)
# voc = ...
# inv_voc = ...


### Task 1.3 — Implementing a HuggingFace-like Tokenizer ⚙

Complete the `A1Tokenizer` class with the following methods:
- `__init__`: store vocabulary and configuration
- `__call__`: encode a list of strings into padded integer tensors (right-side padding)
- `__len__`: return vocabulary size

Also implement the `build_tokenizer` function, and `save()` / `from_file()` for persistence.

**Sanity check:** Verify output tensor shapes and padding behavior match expected format.

In [ ]:
# Task 1.3 — A1Tokenizer class

class A1Tokenizer:
    def __init__(self, voc, inv_voc):
        pass  # TODO

    def __call__(self, texts, padding=True, return_tensors="pt"):
        pass  # TODO

    def __len__(self):
        pass  # TODO

    def save(self, path):
        pass  # TODO

    @classmethod
    def from_file(cls, path):
        pass  # TODO


def build_tokenizer(train_data, val_data, max_voc_size):
    pass  # TODO


---

## Part 2: Data Loading

### Task 2.1 — Loading the texts ⚙

Use HuggingFace Datasets to load from text files and remove empty lines.  
Expected: ~147,000 training and ~18,000 validation instances.

> **Optional:** Create smaller subsets during development using `Subset`.

In [6]:
TRAIN_FILE = "data/wiki.train.tokens"
VAL_FILE   = "data/wiki.valid.tokens"

dataset = load_dataset('text', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})
dataset = dataset.filter(lambda x: x['text'].strip() != '')

print(f"Train: {len(dataset['train'])} | Val: {len(dataset['val'])}")

dataset = load_dataset('text', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})
dataset = dataset.filter(lambda x: x['text'].strip() != '')

# Optional: small subset for faster development
dataset['train'] = Subset(dataset['train'], range(1000))

print(f"Train: {len(dataset['train'])} | Val: {len(dataset['val'])}")

Train: 23767 | Val: 2461
Train: 1000 | Val: 2461


### Task 2.2 — Iterating through the datasets ⚙

Create a PyTorch `DataLoader` with `batch_size` and `shuffle` parameters.  

> **Optional:** Use `DataCollatorForLanguageModeling` for HuggingFace alignment.

**Sanity check:** Examine the first batch output.

In [ ]:
# Task 2.2 — Create DataLoaders

# dl_train = DataLoader(dataset['train'], batch_size=..., shuffle=True)
# dl_val   = DataLoader(dataset['val'],   batch_size=..., shuffle=False)

# Sanity check
# for batch in dl_train:
#     print(batch)
#     break


---

## Part 3: The Neural Network

### Task 3.1 — Setting up the network 🎓

Define `A1RNNModel` inheriting from HuggingFace `PreTrainedModel`.  
The model should include:
- An embedding layer
- An RNN variant (LSTM or GRU) with `batch_first=True`
- An output / unembedding layer

Store hyperparameters in `A1RNNModelConfig` (inheriting from `PretrainedConfig`).

**Sanity check:** Verify no crashes on a test input and output shape is `(batch_size, sequence_length, vocab_size)`.

In [ ]:
# Task 3.1 — Model configuration and architecture

class A1RNNModelConfig(PretrainedConfig):
    def __init__(self, vocab_size=10000, embedding_dim=128, hidden_size=256, **kwargs):
        super().__init__(**kwargs)
        # TODO: store hyperparameters


class A1RNNModel(PreTrainedModel):
    config_class = A1RNNModelConfig

    def __init__(self, config):
        super().__init__(config)
        # TODO: define embedding, RNN, output layers

    def forward(self, input_ids, labels=None):
        # embedded = ...
        # rnn_out, _ = self.rnn(embedded)
        # logits = ...
        pass  # TODO


### Task 3.2 — Computing the loss 🎓

- Exclude the **last position** of logits and the **first position** of labels (next-token prediction)
- Use `CrossEntropyLoss`
- Reshape labels to 1D and logits to 2D before applying the loss

**Sanity check:** Confirm the model produces correct logit shapes without errors.

In [ ]:
# Task 3.2 — Loss computation (integrate into forward() above or implement here)

# shift logits and labels
# shift_logits = logits[:, :-1, :]        # drop last position
# shift_labels = labels[:, 1:]            # drop first position

# reshape for CrossEntropyLoss
# labels_flat  = shift_labels.reshape(-1)
# logits_flat  = shift_logits.reshape(-1, shift_logits.shape[-1])

# loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
# loss = loss_fn(logits_flat, labels_flat)


---

## Part 4: Training

### Task 4.1 — Implementing the trainer 🎓

Complete the `A1Trainer.train()` method:
- Set up an `AdamW` optimizer
- Create training and validation `DataLoader`s
- Implement the training loop monitoring loss
- Set `ignore_index=-100` for padding tokens in `CrossEntropyLoss`
- Replace padding token IDs with `-100` in the labels tensor

> Start with small datasets; full epoch training should take only a few minutes on GPU.

In [ ]:
# Task 4.1 — Trainer

class A1Trainer:
    def __init__(self, model, tokenizer, train_dataset, val_dataset, batch_size=32, lr=1e-3):
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.batch_size = batch_size
        self.lr = lr

    def train(self, n_epochs=5):
        # optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        # loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
        # dl_train = DataLoader(...)
        # dl_val   = DataLoader(...)
        # for epoch in range(n_epochs):
        #     ...training loop...
        #     ...validation loop...
        pass  # TODO


---

## Part 5: Evaluation

### Task 5.1 — Predicting the next word ⚙

Apply the model to integer-encoded text.  
Use `argmax()` or `topk()` on the logits at the **second-to-last position**.  
Convert predicted indices back to strings using the inverse vocabulary.

In [ ]:
# Task 5.1 — Next-word prediction

def predict_next_word(model, tokenizer, text, topk=5):
    pass  # TODO


### Task 5.2 — Computing the perplexity 🎓

Calculate perplexity on the validation set:

$$\text{perplexity} = 2^{-\text{mean log probability}} = \exp(\text{mean cross-entropy loss})$$

**Expected range:** 200–300 for a well-trained model. Values > 700 indicate problems.

> **Optional:** Investigate the effect of different hyperparameters on perplexity.

In [ ]:
# Task 5.2 — Perplexity on validation set

def compute_perplexity(model, tokenizer, val_dataset, batch_size=64):
    pass  # TODO


### Task 5.3 — Inspecting the learned word embeddings 🎓

Find the nearest neighbors of selected words in the embedding space using cosine similarity.  
Verify that neighbors represent semantically similar words.

> **Optional:** Visualize embeddings with a PCA projection to a 2D scatter plot.

In [ ]:
def nearest_neighbors(emb, voc, inv_voc, word, n_neighbors=5):
    test_emb = emb.weight[voc[word]]
    sim_func = nn.CosineSimilarity(dim=1)
    cosine_scores = sim_func(test_emb, emb.weight)
    near_nbr = cosine_scores.topk(n_neighbors + 1)
    topk_cos = near_nbr.values[1:]
    topk_indices = near_nbr.indices[1:]
    return [(inv_voc[ix.item()], cos.item())
            for ix, cos in zip(topk_indices, topk_cos)]

# Task 5.3 — inspect neighbors for a few words
# print(nearest_neighbors(model.embedding, voc, inv_voc, "king"))


In [ ]:
# Optional — PCA visualization

from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

def plot_embeddings_pca(emb, voc, words):
    vectors = np.vstack([emb.weight[voc[w]].cpu().detach().numpy() for w in words])
    vectors -= vectors.mean(axis=0)
    twodim = TruncatedSVD(n_components=2).fit_transform(vectors)
    plt.figure(figsize=(5, 5))
    plt.scatter(twodim[:, 0], twodim[:, 1], edgecolors='k', c='r')
    for word, (x, y) in zip(words, twodim):
        plt.text(x + 0.02, y, word)
    plt.axis('off')
    plt.show()

# plot_embeddings_pca(model.embedding, voc, ["king", "queen", "man", "woman"])
